# 로컬 파일에서 직접 실행하는 백테스트

아래 셀은 **`C:\Python\3.Equity\NoteBook\NoteBook.ipynb`**에 저장된 백테스트 코드를 매번 새로 읽어 실행합니다. 원본 노트북 탭에 남아 있는 편집 내용이나 출력과 무관하게 디스크의 코드를 사용합니다.

실행하면 실제 파일 경로, 코드 해시, 투자금 설정값이 먼저 나오고, 계산이 끝나면 전략과 KOSPI 수익률을 함께 표시합니다.

원본 백테스트 설정을 변경했다면 먼저 원본 파일을 저장한 후 이 셀을 실행하세요. 이 노트북에는 매매 전략을 복사하지 않았습니다.

[직전에 저장한 실제 실행 결과 보고서](Backtest_Result.html)


In [ ]:
# 이 셀은 실행할 때마다 로컬 원본을 디스크에서 새로 읽습니다.
import hashlib
import importlib.util
import json
from datetime import datetime
from pathlib import Path

import FinanceDataReader as fdr
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

LOCAL_ROOT = Path(r"C:\Python\3.Equity")
LOCAL_NOTEBOOK = LOCAL_ROOT / "NoteBook" / "NoteBook.ipynb"
LOCAL_QUOTES = LOCAL_ROOT / "Function" / "KoreanStockQuotes.py"

saved_notebook = json.loads(LOCAL_NOTEBOOK.read_text(encoding="utf-8"))
backtest_cells = [
    cell for cell in saved_notebook["cells"]
    if cell["cell_type"] == "code"
    and "BACKTEST_START =" in "".join(cell.get("source", []))
]
if len(backtest_cells) != 1:
    raise RuntimeError("로컬 원본에서 백테스트 셀을 하나로 식별할 수 없습니다.")
local_source = "".join(backtest_cells[0]["source"])
source_hash = hashlib.sha256(local_source.encode("utf-8")).hexdigest()[:12]

# 지표 함수도 현재 커널에 남은 예전 모듈 대신 로컬 파일에서 불러옵니다.
spec = importlib.util.spec_from_file_location("local_backtest_quotes", LOCAL_QUOTES)
local_quotes = importlib.util.module_from_spec(spec)
spec.loader.exec_module(local_quotes)
analyze_technical_indicators = local_quotes.analyze_technical_indicators

print(f"로컬 원본 직접 실행: {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
print(f"노트북 경로: {LOCAL_NOTEBOOK.resolve()}", flush=True)
print(f"지표 코드 경로: {LOCAL_QUOTES.resolve()}", flush=True)
print(f"실행 코드 SHA256: {source_hash}", flush=True)
for line in local_source.splitlines():
    if line.startswith(("BACKTEST_START =", "BACKTEST_END =", "UNIT_INVESTMENT =", "MAX_INVESTMENT =")):
        print(line, flush=True)
print("위 파일의 저장된 코드로 전체 백테스트를 시작합니다. 수 분 걸릴 수 있습니다.", flush=True)

exec(compile(local_source, str(LOCAL_NOTEBOOK) + ":backtest", "exec"), globals())
